# Lab 07 — RNN, LSTM and GRU for sequence prediction

**Dataset:** [tulipa762/electricity_load_diagrams](https://huggingface.co/datasets/tulipa762/electricity_load_diagrams). We create a small chronological forecasting task and compare recurrent cells under the same window, optimizer and training budget.

## The math behind the experiment

A vanilla RNN carries a hidden state forward:

$$
h_t=\phi(W_xx_t+W_hh_{t-1}+b).
$$

The same $W_h$ is reused at every time step. During backpropagation through time, gradients repeatedly multiply by derivatives associated with $W_h$. If those products become very small, long-range information is hard to learn; if they become very large, gradients can explode.

An LSTM introduces gates to control information flow. A simplified cell update is

$$
c_t=f_t\odot c_{t-1}+i_t\odot\tilde{c}_t.
$$

In [ ]:
!pip -q install datasets torch pandas numpy
from datasets import load_dataset
import numpy as np,pandas as pd,torch
from torch import nn
raw=load_dataset("tulipa762/electricity_load_diagrams","uci",split="train[:1]")
print(raw.features)
row=raw[0]; vals=[]
for v in row.values():
 if isinstance(v,(list,tuple)): vals.extend(v)
 elif isinstance(v,(int,float)): vals.append(v)
vals=np.asarray(vals,dtype=np.float32); vals=vals[np.isfinite(vals)]
vals=(vals-vals.mean())/(vals.std()+1e-6); vals=torch.tensor(vals[:2000])
T=24; X=torch.stack([vals[i:i+T] for i in range(len(vals)-T-1)]); y=vals[T+0:len(vals)-1]
# Align one-step targets.
y=y[:len(X)]; n=int(.8*len(X)); Xtr,Xte,ytr,yte=X[:n],X[n:],y[:n],y[n:]
print(X.shape)

## Step 1 — Prediction

RNNs should learn short temporal structure but can struggle as dependency length grows. LSTM/GRU gates should make long-range credit assignment easier. The comparison must use the same hidden size and training budget.

In [ ]:
class RNNModel(nn.Module):
 def __init__(self,kind="RNN",h=32):
  super().__init__(); R={"RNN":nn.RNN,"LSTM":nn.LSTM,"GRU":nn.GRU}[kind]; self.r=R(1,h,batch_first=True); self.o=nn.Linear(h,1)
 def forward(self,x): return self.o(self.r(x.unsqueeze(-1))[0][:,-1]).squeeze(-1)

def make_windows(series,T):
 X=torch.stack([series[i:i+T] for i in range(len(series)-T-1)])
 y=series[T:T+len(X)]
 return X,y

def fit(kind,T=24,epochs=6):
 m=RNNModel(kind); opt=torch.optim.Adam(m.parameters(),1e-3); loss=nn.MSELoss()
 seq,target=make_windows(vals[:1600],T)
 n=int(.8*len(seq)); Xtr2,ytr2=seq[:n],target[:n]; Xte2,yte2=seq[n:],target[n:]
 for _ in range(epochs):
  opt.zero_grad(); l=loss(m(Xtr2),ytr2); l.backward(); opt.step()
 with torch.no_grad():
  train_mse=loss(m(Xtr2),ytr2).item(); test_mse=loss(m(Xte2),yte2).item()
 return train_mse,test_mse,sum(p.numel() for p in m.parameters())

rows=[]
for k in ["RNN","LSTM","GRU"]:
 rows.append([k,*fit(k,T=24)])
pd.DataFrame(rows,columns=["cell","train_MSE","test_MSE","parameters"])

## Step 2 — Dependency-length intervention

Train/evaluate the same recurrent cell with windows of 8, 24 and 48 steps. If performance degrades as the window grows, that is evidence about optimization/memory—not proof that the architecture “cannot remember.”

In [ ]:
def window_experiment(kind,T):
 seq,target=make_windows(vals[:1600],T); n=int(.8*len(seq)); Xtr2,ytr2=seq[:n],target[:n]; Xte2,yte2=seq[n:],target[n:]
 m=RNNModel(kind); opt=torch.optim.Adam(m.parameters(),1e-3); loss=nn.MSELoss()
 for _ in range(5):
  opt.zero_grad(); l=loss(m(Xtr2),ytr2); l.backward(); opt.step()
 with torch.no_grad(): return loss(m(Xte2),yte2).item()
rows=[[T,window_experiment("RNN",T),window_experiment("LSTM",T),window_experiment("GRU",T)] for T in [8,24,48]]
pd.DataFrame(rows,columns=["window","RNN_test_MSE","LSTM_test_MSE","GRU_test_MSE"])

## Visualize memory — performance vs sequence length

A longer input window gives a model access to more history, but it also changes optimization difficulty and the amount of information the model must use. Compare test error across RNN, LSTM, and GRU cells rather than assuming a gated model always wins.

In [ ]:
import matplotlib.pyplot as plt

window_results = pd.DataFrame(rows, columns=["window", "RNN_test_MSE", "LSTM_test_MSE", "GRU_test_MSE"])
fig, ax = plt.subplots(figsize=(8, 4.8))
for col in ["RNN_test_MSE", "LSTM_test_MSE", "GRU_test_MSE"]:
    ax.plot(window_results["window"], window_results[col], marker="o", label=col.replace("_test_MSE", ""))
ax.set_title("Recurrent cell comparison by context length")
ax.set_xlabel("Input window length (time steps)")
ax.set_ylabel("Held-out MSE")
ax.legend()
ax.grid(alpha=.25)
plt.tight_layout()
plt.show()

## Conclusion / answer key

Expected: the gated models should often be more stable as context length grows, although exact ranking is data- and hyperparameter-dependent. Report what happened rather than asserting a universal LSTM/GRU superiority.

### Research extension
Repeat with chronological train/test evaluation and three seeds; plot error against dependency length.